# TalentIQ - Exploratory Data Analysis

This notebook explores the IBM HR Employee Attrition dataset to understand the data before building any model.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv')
print('Dataset loaded:', df.shape)

## Step 1 - What is the business problem?

A company loses employees every year and hiring replacements costs both time and money. The goal is to predict in advance which employees are likely to leave so the company can take action before it happens.

We are building a model that takes an employee's profile — things like age, salary, and job role — and predicts whether they will leave or stay. This is a binary classification problem where label 1 means the employee left and label 0 means they stayed.

Before building any model, we first need to understand the data. This notebook works through six questions: what the data looks like, whether there are missing values or duplicate rows, whether the classes are balanced, which features seem related to attrition, whether there are outliers, and how the features relate to each other.

## Step 2 - What does the data look like?

We have a CSV file but do not know its shape, column names, or what the values look like. The cells below load it and take a first look at the structure.

In [ ]:
print('Shape:', df.shape)
print('Columns:', list(df.columns))

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

The dataset has 1470 rows and 35 columns. Some columns are numerical like Age and MonthlyIncome, and some are text like Department and Gender. The target column is Attrition with values Yes or No.

## Step 3 - Are there missing values or duplicates?

Missing values will break the model or give wrong results. Duplicate rows can bias it by making it learn the same pattern more than once. The cell below checks for both.

In [ ]:
print('Missing values per column:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\nTotal duplicate rows:', df.duplicated().sum())

No missing values were found — every column is complete. There are also no duplicate rows, so we can move forward without heavy cleaning.

Four columns carry no useful information and will be dropped later: EmployeeCount is always 1, StandardHours is always 80, Over18 is always Y, and EmployeeNumber is just an ID with no relationship to attrition.

## Step 4 - Is the target balanced?

If 95% of employees stayed and only 5% left, a model that always predicts "stay" would be 95% accurate but completely useless. This is called class imbalance and it needs to be checked before training.

In [ ]:
counts = df['Attrition'].value_counts()
pct = df['Attrition'].value_counts(normalize=True) * 100

print('Attrition Counts:')
print(counts)
print('\nPercentage:')
print(pct.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts.plot(kind='bar', ax=ax, color=['steelblue', 'tomato'])
ax.set_title('Class Distribution - Attrition')
ax.set_xlabel('Attrition')
ax.set_ylabel('Count')
ax.set_xticklabels(['No (Stayed)', 'Yes (Left)'], rotation=0)
for i, v in enumerate(counts):
    ax.text(i, v + 5, str(v), ha='center')
plt.tight_layout()
plt.savefig('../reports/figures/class_distribution.png', dpi=150)
plt.show()

About 84% of employees stayed and 16% left. This is a moderately imbalanced dataset. Using accuracy alone as a metric would be misleading here, so we will use F1-macro as the main evaluation metric. We will also apply SMOTE during training to generate synthetic samples for the minority class.

## Step 5 - Which features matter for attrition?

We have 35 columns and need to figure out which ones actually relate to whether someone leaves or stays. Looking at everything randomly would not be efficient. Instead we split features into numerical and categorical and analyze each group separately. For numerical features we compare their distributions between the two groups. For categorical features we check the attrition rate per category.

In [ ]:
numerical_cols = ['Age', 'MonthlyIncome', 'TotalWorkingYears', 'YearsAtCompany',
                  'YearsWithCurrManager', 'DistanceFromHome', 'NumCompaniesWorked',
                  'YearsSinceLastPromotion', 'PercentSalaryHike']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    for label, color in zip(['No', 'Yes'], ['steelblue', 'tomato']):
        axes[i].hist(df[df['Attrition'] == label][col], bins=20,
                     alpha=0.6, color=color, label=label)
    axes[i].set_title(col)
    axes[i].legend(title='Left')

plt.suptitle('Numerical Feature Distributions by Attrition', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/numerical_distributions.png', dpi=150)
plt.show()

Looking at the numerical features, younger employees are more likely to leave. Employees with lower monthly income tend to leave more, and so do those with fewer total working years or fewer years at the current company. Distance from home has some effect but it is weaker than the others. Employees who have worked at more companies before also tend to leave more.

In [ ]:
categorical_cols = ['Department', 'JobRole', 'MaritalStatus', 'OverTime',
                    'BusinessTravel', 'Gender', 'EducationField']

fig, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    attrition_rate = df.groupby(col)['Attrition'].apply(
        lambda x: (x == 'Yes').mean() * 100
    ).sort_values(ascending=False)
    attrition_rate.plot(kind='bar', ax=axes[i], color='tomato')
    axes[i].set_title(f'Attrition Rate by {col}')
    axes[i].set_ylabel('Attrition Rate (%)')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=30, ha='right')

fig.delaxes(axes[-1])
plt.tight_layout()
plt.savefig('../reports/figures/categorical_attrition_rates.png', dpi=150)
plt.show()

For categorical features, OverTime is the strongest signal — employees who do overtime leave at a much higher rate. Single employees leave more than married or divorced ones. Sales Representatives have the highest attrition rate among all job roles. Frequent travelers also leave more. Gender has a small effect and is weaker compared to the other categorical features.

## Step 6 - Are there outliers in numerical columns?

Extreme values can distort what the model learns. If one employee earns ten times more than everyone else, that single row can skew the model toward income. Box plots give a quick visual check for this.

In [ ]:
outlier_cols = ['MonthlyIncome', 'TotalWorkingYears', 'YearsAtCompany', 'DistanceFromHome']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for i, col in enumerate(outlier_cols):
    axes[i].boxplot(df[col].dropna())
    axes[i].set_title(col)

plt.suptitle('Outlier Check - Box Plots', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/figures/boxplots.png', dpi=150)
plt.show()

MonthlyIncome and TotalWorkingYears both have visible outliers on the high end, and YearsAtCompany has some employees with very long tenure. These will be clipped using the IQR method during preprocessing to keep extreme values from distorting the model.

## Step 7 - How are numerical features related to each other?

If two features are highly correlated — for example Age and TotalWorkingYears — they carry the same information. This is called multicollinearity and it can confuse some models. A correlation heatmap shows how strongly each pair of numerical features is related.

In [ ]:
num_df = df.select_dtypes(include='number').drop(
    columns=['EmployeeCount', 'StandardHours', 'EmployeeNumber'], errors='ignore'
)

# Map Attrition to numeric for correlation
num_df['Attrition'] = (df['Attrition'] == 'Yes').astype(int)

corr = num_df.corr()

plt.figure(figsize=(16, 12))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, annot_kws={'size': 7})
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('../reports/figures/heatmap.png', dpi=150)
plt.show()

In [ ]:
# Show top correlations with the target
target_corr = corr['Attrition'].drop('Attrition').sort_values(key=abs, ascending=False)
print('Top features correlated with Attrition:')
print(target_corr.head(10).round(3))

Age, TotalWorkingYears, YearsAtCompany, and YearsWithCurrManager are all strongly correlated with each other, which makes sense since they all measure time and experience. MonthlyIncome is also correlated with Age and TotalWorkingYears because older and more experienced employees tend to earn more. The features most correlated with Attrition are YearsWithCurrManager, TotalWorkingYears, YearsAtCompany, and MonthlyIncome — all negative correlations, meaning more years and more income means less likely to leave.

## Summary - What did EDA tell us?

The dataset had no missing values and no duplicates so no heavy cleaning was needed. The target is imbalanced at 84% vs 16%, which means we will use F1-macro as the metric and apply SMOTE during training. OverTime turned out to be the strongest single predictor and will be kept as a feature. MonthlyIncome, Age, and the years-based columns all matter and will be kept, along with some new combined features engineered from them. Outliers in MonthlyIncome and TotalWorkingYears will be clipped using IQR. The constant columns EmployeeCount, StandardHours, and Over18 will be dropped during preprocessing.

The next step is preprocessing followed by feature engineering, where we act on all of these findings.